# Jamaica terrestrial flooding analysis

- Intersect Jamaica land cover with catchments

In [ ]:
import os
import re
from glob import glob
from itertools import chain

import geopandas
import pandas
import fiona
import networkx as nx
import numpy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
from analysis_utils import *

In [ ]:
base_path = 'L:\\jamaica\\Inputs'

In [ ]:
output_path = 'L:\\jamaica\\Results'

In [ ]:
jamaica_crs = 3448

In [ ]:
jamaicaboundary = geopandas.read_file(os.path.join(base_path, 'jamaica.gpkg'))

In [ ]:
jamaicaboundary

In [ ]:
jamaicaboundary = jamaicaboundary.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system

In [ ]:
jamaicaboundary['area_hectares'] = 0.0001*jamaicaboundary.geometry.area # Convert area to hectares

In [ ]:
jamaica_total_area = jamaicaboundary['area_hectares'].sum()

In [ ]:
layer_list = ['Landcover/2013_landuse_landcover.gpkg','Shapefiles/hydrobasins.gpkg']
layer_name = ["Landuse", "Hydrobasins"]
layer_info = list(zip(layer_list,layer_name))
area_outputs = []

In [ ]:
for idx,(layer,layer_name) in enumerate(layer_info):
    data = geopandas.read_file(os.path.join(base_path, layer))
    data = data.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
    data['area_hectares'] = 0.0001*data.geometry.area # Convert area to hectares
    total_area = data['area_hectares'].sum()
    area_outputs.append((layer_name,total_area,100.0*total_area/jamaica_total_area))
    if layer == 'Landcover/2013_landuse_landcover.gpkg':
        land_use_area = data[['area_hectares', 'Classify']].groupby('Classify').sum()
        land_use_area["area_percentage"] = 100.0*land_use_area["area_hectares"]/jamaica_total_area
        land_use_area.to_csv(os.path.join(output_path, '2013_landuse_landcover_areas2.csv'))
    

In [ ]:
area_outputs

In [ ]:
fiona.listlayers("L:\\jamaica/Inputs/Shapefiles/hydrobasins.gpkg")

In [ ]:
hydrobasins = geopandas.read_file(os.path.join(base_path, 'Shapefiles/hydrobasins.gpkg'), layer='hybas_lake_na_lev12_v1c')
#hydrobasins = hydrobasins.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system - haven't we already done this above?

In [ ]:
len(hydrobasins)

In [ ]:
len(hydrobasins.HYBAS_ID.unique())

In [ ]:
landcover = geopandas.read_file(os.path.join(base_path, 'Landcover/2013_landuse_landcover.gpkg'))[["OBJECTID","geometry","Classify"]]
#landcover = landcover.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system haven't we already done this above?

In [ ]:
hydrobasins['area'] = hydrobasins.geometry.area

In [ ]:
hydrobasins_with_landcover = hydrobasins.overlay(landcover, how='intersection')

In [ ]:
hydrobasins_with_landcover = hydrobasins_with_landcover[['HYBAS_ID', 'Classify', 'geometry']]

In [ ]:
hydrobasins_with_landcover['hydrobasins_landcover_area'] = hydrobasins_with_landcover.geometry.area

In [ ]:
hydrobasins_with_landcover.to_file(
    os.path.join(output_path, "river_flooding_analysis", "hydrobasin_landcover_intersection.gpkg"),
    driver="GPKG")

In [ ]:
hydrobasins_intersection_area = hydrobasins_with_landcover[['HYBAS_ID', 'hydrobasins_landcover_area']].groupby('HYBAS_ID').sum()
hydrobasins_intersection_area

In [ ]:
hydrobasins_with_landcover_area = hydrobasins_with_landcover.groupby(['HYBAS_ID', 'Classify']).sum().reset_index()

In [ ]:
hydrobasins_with_landcover_area = hydrobasins_with_landcover_area.merge(hydrobasins[['HYBAS_ID', 'area']])

In [ ]:
hydrobasins_with_landcover_area['Classify'].unique()

In [ ]:
landuse_mapping = {
    'Bamboo and Secondary Forest': [
        ('forest', 0.5),
        ('afforestable', 0.5)
    ],
    'Buildings and other infrastructures': [
        ('urban', 1.0)
    ],
    'Disturbed broadleaved forest (Secondary Forest)': [
        ('forest', 1.0)
    ],
    'Fields and Secondary Forest': [
        ('forest', 0.5),
        ('agriculture', 0.5)
    ],
    'Fields: Herbaceous crops, fallow, cultivated vegetables': [
        ('agriculture', 1.0)
    ],
    'Herbaceous Wetland': [
        ('non_afforestable', 1.0)
    ],
    'Plantation: Tree crops, shrub crops, sugar cane, banana': [
        ('agriculture', 1.0)
    ],
    'Quarry': [
        ('afforestable', 1.0)
    ],
    'Secondary Forest': [
        ('forest', 1.0)
    ],
    'Water Body': [
        ('non_afforestable', 1.0)
    ],
    'Bamboo and Fields': [
        ('afforestable', 0.5),
        ('agriculture', 0.5)
    ],
    'Bare Rock': [
        ('non_afforestable', 1.0)
    ],
    'Closed broadleaved forest (Primary Forest)': [
        ('forest', 1.0)
    ],
    'Fields  and Bamboo': [
        ('afforestable', 0.5),
        ('agriculture', 0.5)
    ],
    'Fields or Secondary Forest/Pine Plantation': [
        ('forest', 0.5),
        ('agriculture', 0.5)
    ],
    'Fields: Pasture,Human disturbed, grassland': [
        ('afforestable', 1.0)
    ],
    'Fields: Bare Land': [
        ('afforestable', 1.0)
    ],
    'Mangrove Forest': [
        ('non_afforestable', 1.0),
    ],
    'Open dry forest - Tall (Woodland/Savanna)': [
        ('forest', 1.0)
    ],
    'Hardwood Plantation: Euculytus': [
        ('forest', 1.0),
    ],
    'Hardwood Plantation: Mixed': [
        ('forest', 1.0),
    ],
    'Open dry forest - Short': [
        ('forest', 1.0),
    ],
    'Bauxite Extraction': [
        ('afforestable', 1.0),
    ],
    'Swamp Forest': [
        ('non_afforestable', 1.0),
    ],
    'Bamboo': [
        ('afforestable', 1.0),
    ],
    'Hardwood Plantation: Mahogany': [
        ('forest', 1.0),
    ],
    'Hardwood Plantation: Mahoe': [
        ('forest', 1.0),
    ]
}

## Percentages

In [ ]:
hydrobasins_with_landcover_area['hydrobasins_landcover_area_perc'] = 100 * hydrobasins_with_landcover_area.hydrobasins_landcover_area / hydrobasins_with_landcover_area.area

In [ ]:
hydrobasins_with_landcover_area.head(2)

In [ ]:
hydrobasins_with_landcover_area_pivot = hydrobasins_with_landcover_area[['HYBAS_ID', 'Classify', 'hydrobasins_landcover_area']] \
    .pivot(columns="Classify", index="HYBAS_ID", values="hydrobasins_landcover_area") \
    .fillna(0)

In [ ]:
hydrobasins_with_landcover_area_pivot.columns

In [ ]:
def calculate_reclassified_area(df, mapping):
    output = df.reset_index()[['HYBAS_ID']].copy()
    
    # find the set of output classes
    classes = set()
    for landuse, reclass_proportions in mapping.items():
        for reclass, proportion in reclass_proportions:
            classes.add(reclass)
    classes = sorted(list(classes))
    
    # set up empty values
    for reclass in classes:
        output[reclass] = 0
        
    # add landuse area
    for landuse, reclass_proportions in mapping.items():
        for reclass, proportion in reclass_proportions:
            output[reclass] += (df[landuse].values * proportion)
            
    return output

hydrobasins_reclassified_area = calculate_reclassified_area(hydrobasins_with_landcover_area_pivot, landuse_mapping)

In [ ]:
hydrobasins_reclassified_area.drop(columns='HYBAS_ID').plot(kind='area')

In [ ]:
hydrobasins_reclassified_area_perc = hydrobasins_reclassified_area.set_index('HYBAS_ID').join(
    hydrobasins[['HYBAS_ID', 'area', 'geometry']].set_index('HYBAS_ID')
).join(hydrobasins_intersection_area)  # use the intersection for "total" basin area - so that our landuse sums to 100%

for reclass in [c for c in hydrobasins_reclassified_area.columns if c != 'HYBAS_ID']:
    hydrobasins_reclassified_area_perc[f"{reclass}_perc"] = \
        (hydrobasins_reclassified_area_perc[reclass] / hydrobasins_reclassified_area_perc.hydrobasins_landcover_area) * 100
    
hydrobasins_reclassified_area_perc.reset_index() \
    [[c for c in hydrobasins_reclassified_area_perc.columns if "_perc" in c]] \
    .plot(kind='area')

In [ ]:
geopandas.GeoDataFrame(hydrobasins_reclassified_area_perc).to_file(
    os.path.join(output_path, "river_flooding_analysis","hydrobasins_with_reclassified_landcover.gpkg"), driver="GPKG")

In [ ]:
hydrobasins_reclassified_area_perc.drop(columns="geometry").to_csv(
 os.path.join(output_path, "river_flooding_analysis","hydrobasins_with_reclassified_landcover.csv"))

In [ ]:
hydrobasins_reclassified_area_perc.loc[7120065560]

In [ ]:
all_uses = list(hydrobasins_with_landcover_area.Classify.unique())

In [ ]:
def calculate_forest_area(row):
    forested_classes = [
        'Closed broadleaved forest (Primary Forest)',
        'Disturbed broadleaved forest (Secondary Forest)',
        'Secondary Forest',
    ]
    half_forested_classes = [
        'Bamboo and Secondary Forest',
        'Fields and Secondary Forest',
        'Fields or Secondary Forest/Pine Plantation',
    ]
    
    #check the literature for what the half forested classes actually mean - it should be in my notes already
    if row.Classify in forested_classes :
        return row.hydrobasins_landcover_area
    
    if row.Classify in half_forested_classes :
        return row.hydrobasins_landcover_area / 2

    return 0

hydrobasins_with_landcover_area["forest_area"] = hydrobasins_with_landcover_area.apply(calculate_forest_area, axis=1)
hydrobasin_forest_cover = (
    hydrobasins_with_landcover_area
    .groupby(["HYBAS_ID", "area"])
    .sum(numeric_only=True)
    [["forest_area"]]
    .reset_index()
)
hydrobasin_forest_cover["forest_perc"] = 100 * hydrobasin_forest_cover.forest_area / hydrobasin_forest_cover.area

In [ ]:
# HYBAS_ID, forest_perc, forest_area, total_area
hydrobasin_forest_cover

In [ ]:
csv_path = os.path.join(output_path, "river_flooding_analysis", "hydrobasin_forest_cover.csv")
gpkg_path = os.path.join(output_path, "river_flooding_analysis", "hydrobasin_forest_cover.gpkg")

hydrobasin_forest_cover.to_csv(csv_path, index=False)
hydrobasin_forest_cover_gdf = geopandas.GeoDataFrame(hydrobasin_forest_cover.set_index("HYBAS_ID").join(
    hydrobasins[["HYBAS_ID", "geometry"]].copy().set_index("HYBAS_ID")))
hydrobasin_forest_cover_gdf.to_file(gpkg_path, driver="GPKG")

In [ ]:
hydrobasin_forest_cover.HYBAS_ID = hydrobasin_forest_cover.reset_index().HYBAS_ID.astype("str")
hydrobasin_forest_cover.sort_values(by="forest_perc").plot(x="HYBAS_ID", y="forest_perc", kind="scatter")

In [ ]:
hydrobasins_all_cover = hydrobasins_with_landcover_area_pivot.join(hydrobasin_forest_cover_gdf)

# calculate percentages
for use in all_uses:
    hydrobasins_all_cover[f"{use}_perc"] = (hydrobasins_all_cover[use] / hydrobasins_all_cover.area) * 100
    
hydrobasins_all_cover.head()

In [ ]:
csv_path = os.path.join(output_path, "river_flooding_analysis", "hydrobasin_all_cover.csv")
gpkg_path = os.path.join(output_path, "river_flooding_analysis", "hydrobasin_all_cover.gpkg")

hydrobasins_all_cover.drop(columns="geometry").to_csv(csv_path)
geopandas.GeoDataFrame(hydrobasins_all_cover).to_file(gpkg_path, driver="GPKG")

# Basin network

In [ ]:
def create_basin_network(hydrobasins):
    basin_network = nx.DiGraph()
    basin_network.add_node(0) # add final sink (the sea)
    basin_network.add_nodes_from(hydrobasins.HYBAS_ID) # add all basins as nodes
    for basin in hydrobasins.itertuples():
        # add directed edges from sink, pointing upstream to source
        basin_network.add_edge(basin.NEXT_DOWN, basin.HYBAS_ID)
    return basin_network

In [ ]:
def find_upstream_basin_ids(basin_network, basin_id):
    successors = nx.bfs_successors(basin_network, basin_id)
    all_upstream = [basin_id]
    for basin, basin_ids in successors:
        all_upstream.extend(basin_ids)
    return all_upstream

In [ ]:
def select_upstream_basins(hydrobasins, basin_ids):
    return hydrobasins[hydrobasins.HYBAS_ID.isin(basin_ids)]

In [ ]:
basin_network = create_basin_network(hydrobasins)

In [ ]:
basin_network.number_of_nodes(), basin_network.number_of_edges()

In [ ]:
list(basin_network.successors(7120852530)), list(basin_network.successors(7120065210))

In [ ]:
find_upstream_basin_ids(basin_network, 7120852530)

In [ ]:
find_upstream_basin_ids(basin_network, 7120065210)

In [ ]:
for hybas_id in hydrobasins.HYBAS_ID:
    upstream_ids = find_upstream_basin_ids(basin_network, hybas_id)
    upstream_ids.append(hybas_id)
    df = select_upstream_basins(hydrobasins_with_landcover_area, upstream_ids)
    break
df

In [ ]:
select_upstream_basins(hydrobasins, find_upstream_basin_ids(basin_network, 7120065210))

In [ ]:
nx.draw(basin_network, with_labels=True)

# Forests in relation to flooded infrastructure

In [ ]:
glob('L:\\Jamaica\\processed_data\\*.parquet')

#### Infrastructure exposure and damages

- read in infrastructure exposure (depths/windspeeds)
- calculate damages (at return periods)
- calculate expected annual damages

In [ ]:
df = geopandas.read_parquet('L:\\Jamaica\\processed_data\\airport_polygon_with_fluvial.parquet')

In [ ]:
list(df.columns)

In [ ]:
fluvial_cols = [col for col in df.columns if col.startswith("fluvial")]
surface_cols = [col for col in df.columns if col.startswith("surface")]
coastal_cols = [col for col in df.columns if col.startswith("coastal")]
hazard_cols = fluvial_cols
for col in hazard_cols:
    mask = (df[col] < 0)
    df.loc[mask, col] = 0
    
df.fillna(0, inplace=True)
max_flood_depth = df[hazard_cols].max(axis=1)

flooded_df = df[max_flood_depth > 0].copy()

In [ ]:
flooded_df

In [ ]:
class DamageCurve():
    """A piecewise-linear damage curve"""

    def __init__(self, curve):
        curve = curve.copy()
        self.intensity, self.damage = curve.intensity, curve.damage

        bounds = (self.damage.min(), self.damage.max())
        self._interpolate = interp1d(
            self.intensity,
            self.damage,
            kind="linear",
            fill_value=bounds,
            bounds_error=False,
            copy=False,
        )

    def damage_fraction(self, exposure: numpy.array) -> numpy.array:
        """Evaluate damage fraction for exposure to a given hazard intensity"""
        return self._interpolate(exposure)

In [ ]:
damage_curve_files = glob("../../damage_curves/*.xlsx")
damage_curve_files

In [ ]:
damage_curve_lookup = {}

for fname in damage_curve_files:
    sector, hazard_type = re.search(r"damage_curves_([^_]+)_([^_]+).xlsx", fname).groups()
    sheets = pandas.read_excel(fname, sheet_name=None)
    sheet_names = set(sheets.keys()) ^ set(['Sources'])
    for sheet_name in sheet_names:
        data = sheets[sheet_name]
        damage_curve_lookup[sector, hazard_type, sheet_name] = data


In [ ]:
# for key in damage_curve_lookup.keys():
#     print(key)

In [ ]:
pandas.read_csv( '../../damage_curves\\hazard_damage_parameters.csv')

In [ ]:
asset_damage_curve_mapping = pandas.read_csv('../../damage_curves\\asset_damage_curve_mapping.csv')
asset_sheet_for_asset_name = {}
for row in asset_damage_curve_mapping.itertuples():
    asset_sheet_for_asset_name[row.sector, row.hazard_type, row.asset_name] = row.asset_sheet

# for key in asset_sheet_for_asset_name.keys():
#     print(key)

In [ ]:
asset_sheet_for_asset_name[('transport', 'flooding', 'runway')]

In [ ]:
asset_details = pandas.read_csv('../../Inputs/networks/network_layers_hazard_intersections_details.csv')
asset_details

In [ ]:
flooded_df.asset_type.unique()

In [ ]:
damage_curve = DamageCurve(pandas.DataFrame(
    {"intensity": [0.0, 10, 20, 30], "damage": [0, 0.1, 0.2, 1.0]}
))

In [ ]:
flooded_df.head()

In [ ]:
a

In [ ]:
damage_fraction_rp20 = damage_curve.damage_fraction(flooded_df["fluvial__rp_20__rcp_baseline__epoch_2010__conf_None"])

In [ ]:
damage_fraction_rp20 * 1_000_000_000

In [ ]:
def basin_id_for_point(point, hydrobasins):
    return hydrobasins.loc[hydrobasins.contains(point), "HYBAS_ID"].iloc[0]

In [ ]:
def calculate_basin_forest_perc(row):
    basin_id = basin_id_for_point(row.geometry, hydrobasins)
    basin_ids = find_upstream_basin_ids(basin_network, basin_id)
    point_basin_cover = hydrobasin_forest_cover[hydrobasin_forest_cover.HYBAS_ID.isin(basin_ids)]
    forest_perc = 100 * point_basin_cover.forest_area.sum() / point_basin_cover.area.sum()
    return forest_perc

flooded_df["upstream_basin_forest_perc"] = flooded_df.apply(calculate_basin_forest_perc, axis=1)
flooded_df[flooded_df.upstream_basin_forest_perc > 40]

In [ ]:
fig, ax = plt.subplots()
point_basins.plot(ax=ax)
flooded_df.plot(ax=ax, color="red")

# Rivers

In [ ]:
major_rivers = geopandas.read_file("Z:\\jamaica/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="Major_Rivers")
headwater_rivers = geopandas.read_file("Z:\\jamaica/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="ja_fw_target_headwater_streams_18jan06")

In [ ]:
headwater_rivers_and_major_rivers = major_rivers.overlay(headwater_rivers, how='union')

In [ ]:
headwater_rivers_and_major_rivers = headwater_rivers_and_major_rivers.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system

In [ ]:

figure, ax = plot.subplots()
hydrobasins.plot(ax=ax, column='MAIN_BAS', cmap='tab20', legend='true')
headwater_rivers_and_major_rivers.plot()
ax

In [ ]:
hydrobasins_MAIN_BAS = hydrobasins.MAIN_BAS.apply(str)
#hydrobasins.MAIN_BAS = hydrobasins.MAIN_BAS.apply(str)
hydrobasins.plot(column='MAIN_BAS', cmap='tab20', legend='true')

In [ ]:
major_rivers = geopandas.read_file("Z:\\jamaica/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="Major_Rivers")

In [ ]:
major_rivers.plot()

In [ ]:
headwater_rivers = geopandas.read_file("Z:\\jamaica/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="ja_fw_target_headwater_streams_18jan06")

In [ ]:
headwater_rivers.plot()

In [ ]:
headwater_rivers_and_major_rivers = major_rivers.overlay(headwater_rivers, how='union')

In [ ]:
headwater_rivers_and_major_rivers = headwater_rivers_and_major_rivers.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system

In [ ]:
headwater_rivers_and_major_rivers.plot()

In [ ]:
headwater_rivers_and_major_rivers.to_file(os.path.join(output_path, f'headwater_rivers_and_major_rivers.gpkg'),driver="GPKG")

In [ ]:
hydrobasins_with_headwater_rivers_and_major_rivers = hydrobasins_MAIN_BAS \
    .overlay(headwater_rivers_and_major_rivers, how='intersection')

landcover_bauxite_allprotected = landcover_bauxite \
    .overlay(all_protected_areas.set_geometry("geometry"), how='intersection')

In [ ]:
hydrobasins_with_headwater_rivers_and_major_rivers.plot()

In [ ]:
hydrobasins.MAIN_BAS = hydrobasins.MAIN_BAS.apply(str)
hydrobasins.plot(column='MAIN_BAS', cmap='tab20', legend='true')